<a href="https://colab.research.google.com/github/rmadatt/ADLAB/blob/main/Navier_Stokes_OpenAI_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Navier-Stokes 2D Fluid Animation - OpenAI-style Visualization

This is a **pure Python** solver for the incompressible Navier-Stokes equations:

**du/dt + (u·∇)u = -∇p + ν∇²u + F**

We visualize a 2D fluid like smoke. Runs in Colab, no CUDA needed.

Inspired by Jos Stam Stable Fluids (used in games) and OpenAI's neural fluid demos, but solved with classic finite differences.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Grid size - keep small for Colab speed
Nx, Ny = 64, 64
dt = 0.1
nu = 0.1  # viscosity
steps = 300

# Fields
u = np.zeros((Ny, Nx))  # x-velocity
v = np.zeros((Ny, Nx))  # y-velocity
u_prev = np.zeros((Ny, Nx))
v_prev = np.zeros((Ny, Nx))

def diffuse(x, x0, a):
    # Simple Gauss-Seidel diffusion
    for _ in range(4):
        x[1:-1,1:-1] = (x0[1:-1,1:-1] + a*(x[2:,1:-1]+x[:-2,1:-1]+x[1:-1,2:]+x[1:-1,:-2]))/(1+4*a)
    return x

def advect(d, d0, u, v, dt):
    # Semi-Lagrangian advection
    j, i = np.meshgrid(np.arange(Ny), np.arange(Nx), indexing='ij')
    x = i - dt*u
    y = j - dt*v
    x = np.clip(x, 0.5, Nx-1.5)
    y = np.clip(y, 0.5, Ny-1.5)
    i0 = x.astype(int)
    j0 = y.astype(int)
    i1 = i0+1
    j1 = j0+1
    s1 = x - i0
    t1 = y - j0
    s0 = 1-s1
    t0 = 1-t1
    d[:,:] = s0*t0*d0[j0,i0] + s1*t0*d0[j0,i1] + s0*t1*d0[j1,i0] + s1*t1*d0[j1,i1]
    return d

def project(u, v, p, div):
    div[1:-1,1:-1] = -0.5*((u[1:-1,2:]-u[1:-1,:-2]) + (v[2:,1:-1]-v[:-2,1:-1]))/2
    p[:,:]=0
    for _ in range(10):
        p[1:-1,1:-1] = (div[1:-1,1:-1] + p[2:,1:-1]+p[:-2,1:-1]+p[1:-1,2:]+p[1:-1,:-2])/4
    u[1:-1,1:-1] -= 0.5*(p[1:-1,2:]-p[1:-1,:-2])/2
    v[1:-1,1:-1] -= 0.5*(p[2:,1:-1]-p[:-2,1:-1])/2
    return u,v

# Storage for animation
frames = []
p = np.zeros((Ny,Nx))
div = np.zeros((Ny,Nx))

print('Simulating Navier-Stokes...')
for n in range(steps):
    # Add force - like stirring in center (openAI-style jet)
    if n < 100:
        u[Ny//2-5:Ny//2+5, Nx//2-5:Nx//2+5] += 2.0
        v[Ny//4, Nx//4:Nx//2] += 1.0
    if 100 <= n < 150:
        u[Ny//2, Nx//4:Nx//2] -= 1.5

    u_prev, v_prev = u.copy(), v.copy()

    # Diffusion
    a = dt*nu*Nx*Ny
    u = diffuse(u, u_prev, a)
    v = diffuse(v, v_prev, a)

    # Projection
    u, v = project(u, v, p, div)

    # Advection
    u_prev, v_prev = u.copy(), v.copy()
    u = advect(u, u_prev, u_prev, v_prev, dt)
    v = advect(v, v_prev, u_prev, v_prev, dt)

    # Projection again
    u, v = project(u, v, p, div)

    if n % 5 == 0:
        vel = np.sqrt(u**2 + v**2)
        frames.append(vel.copy())
        if n % 50 == 0:
            print(f'Step {n}/{steps}')

print(f'Done, {len(frames)} frames')


Simulating Navier-Stokes...
Step 0/300
Step 50/300
Step 100/300
Step 150/300
Step 200/300
Step 250/300
Done, 60 frames


In [2]:
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(frames[0], cmap='inferno', vmin=0, vmax=2, origin='lower')
ax.set_title('Navier-Stokes: Velocity Magnitude | ν=0.1')
ax.set_xticks([])
ax.set_yticks([])
plt.colorbar(im, ax=ax, label='Speed')

def update(k):
    im.set_data(frames[k])
    ax.set_title(f'Navier-Stokes Fluid - Step {k*5} | OpenAI-style visualization')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=50, blit=True)
plt.close()
HTML(ani.to_jshtml())


### What you are seeing

- **Color = speed** (inferno: black slow, yellow fast)
- **Equations solved**: Incompressible Navier-Stokes
  - `∂u/∂t + (u·∇)u = -∇p + ν∇²u`
  - `∇·u = 0`
- **Method**: Stable Fluids (diffuse -> project -> advect -> project)
- **Viscosity ν=0.1** = thick fluid like honey. Lower it to 0.01 for turbulent smoke.

Change the force lines to stir differently and you get different flows — this is how OpenAI's fluid demos start, before replacing the solver with a neural network.
